# 🎙️ Douyin Vietnamese Dubbing Server

Extension sẽ tự bấm **Run all**. Nếu tự động không hoạt động, bấm **Runtime → Run all** một lần. Giữ tab này mở trong lúc xử lý video. Notebook dùng GPU Colab, FastAPI và Cloudflare Quick Tunnel tạm thời.

In [ ]:
# Cài mã nguồn và dependency. Cell này có thể chạy lại an toàn.
import hashlib, json, os, pathlib, shutil, subprocess, sys
print('NEKO_PROGRESS ' + json.dumps({'stage':'install','progress':20,'message':'Đang cài thư viện AI trên Colab · lần đầu có thể mất vài phút…'}, ensure_ascii=False), flush=True)
ROOT = pathlib.Path('/content/split-video')
if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Dattrong0512/split-video.git', str(ROOT)], check=True)
requirements = ROOT / 'backend/requirements-colab.txt'
dependency_hash = hashlib.sha256(requirements.read_bytes()).hexdigest()
dependency_marker = pathlib.Path('/content/.douyin-dubbing-dependencies')
if not dependency_marker.exists() or dependency_marker.read_text() != dependency_hash:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', str(requirements)], check=True)
        dependency_marker.write_text(dependency_hash)
    except Exception as error:
        print('INSTALL_FAILED: ' + str(error), flush=True)
        raise
else:
    print('NEKO_PROGRESS ' + json.dumps({'stage':'cached','progress':48,'message':'Đã dùng lại thư viện AI trong runtime Colab hiện tại.'}, ensure_ascii=False), flush=True)
if not shutil.which('cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb', '-O', '/tmp/cloudflared.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True, stdout=subprocess.DEVNULL)
sys.path.insert(0, str(ROOT))
print('NEKO_PROGRESS ' + json.dumps({'stage':'installed','progress':55,'message':'Đã cài thư viện. Đang khởi động máy chủ…'}, ensure_ascii=False), flush=True)

In [ ]:
# Khởi động API và tunnel; extension tự đọc dòng NEKO_SERVER_READY.
import json, os, queue, re, secrets, subprocess, threading, time, urllib.request
print('NEKO_PROGRESS ' + json.dumps({'stage':'gpu','progress':60,'message':'Đã kết nối GPU T4. Đang khởi động API…'}, ensure_ascii=False), flush=True)
import torch, uvicorn
if not torch.cuda.is_available():
    raise RuntimeError('GPU_UNAVAILABLE: Runtime → Change runtime type → T4 GPU, sau đó Run all lại.')
session_token = secrets.token_urlsafe(32)
os.environ['DUBBING_SESSION_TOKEN'] = session_token
os.environ['DUBBING_WORK_ROOT'] = '/content/douyin-dubbing-jobs'
import backend.server as server_module
server_module.SESSION_TOKEN = session_token
try:
    ready_request = urllib.request.Request('http://127.0.0.1:7860/api/health', headers={'Authorization': 'Bearer ' + session_token})
    urllib.request.urlopen(ready_request, timeout=1)
    api_thread = None
except Exception:
    api_thread = threading.Thread(target=lambda: uvicorn.run(server_module.app, host='127.0.0.1', port=7860, log_level='warning'), daemon=True)
    api_thread.start()
api_ready = False
for _ in range(60):
    try:
        ready_request = urllib.request.Request('http://127.0.0.1:7860/api/health', headers={'Authorization': 'Bearer ' + session_token})
        urllib.request.urlopen(ready_request, timeout=1)
        api_ready = True
        break
    except Exception:
        time.sleep(1)
if not api_ready:
    raise RuntimeError('API_FAILED: Không khởi động được API trong Colab.')
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
    try: tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired: tunnel.kill()
print('NEKO_PROGRESS ' + json.dumps({'stage':'tunnel','progress':82,'message':'Đang tạo kết nối Cloudflare bảo mật…'}, ensure_ascii=False), flush=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tunnel_lines = queue.Queue()
def read_tunnel_output():
    for output_line in iter(tunnel.stdout.readline, ''):
        tunnel_lines.put(output_line)
_tunnel_reader = threading.Thread(target=read_tunnel_output, daemon=True)
_tunnel_reader.start()
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    try: line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        if tunnel.poll() is not None: break
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('TUNNEL_FAILED: Không tạo được Cloudflare Quick Tunnel.')
server_module.PUBLIC_URL = public_url
handshake = {'url': public_url, 'token': session_token, 'createdAt': int(time.time())}
print('NEKO_SERVER_READY ' + json.dumps(handshake, separators=(',', ':')), flush=True)
print('✅ Máy chủ sẵn sàng. Giữ tab này mở; quay lại Douyin để tiếp tục.')
# Thread đọc log tiếp tục chạy nền để cloudflared không bị nghẽn stdout.
# Cell kết thúc ở đây; uvicorn thread và cloudflared process vẫn chạy trong runtime Colab.